# Hydrocarbon Rock Segmentation Under UV Fluorescence
**AI-powered microscopic image segmentation** based on exact HSV color thresholding.

### Reference Literature
This implementation strictly adheres to the scientific constraints established in the article *"Novel Lithology Identification Method for Drilling Cuttings Under PDC Bit"*:
1.  **Color Space Conversion**: Utilizes HSV formatting to effectively isolate Hue from light/intensity variance (e.g., reflections from metallic trays).
2.  **Color Thresholding**: Uses specialized predefined boundaries capturing UV luminescence (Cyan & Yellow) for Hydrocarbon.
3.  **Area Filtering Constraint**: Validates a fluorescent contour as a hydrocarbon rock exclusively if its contour area $\ge 100$ pixels, neutralizing structural noise.

In [ ]:
import numpy as np
print(np.__version__)  

import cv2
print(cv2.__version__) 

import matplotlib.pyplot as plt
from PIL import Image

# widget
import ipywidgets as widgets

import io
import sys
import os
from IPython.display import display, clear_output


# Import the color configuration
from color_config import COLOR_THRESHOLDS

pixel_threshold = 10 #define threshold area in pixels


1.26.4
4.13.0


In [4]:
def segment_hydrocarbon_uv(image_rgb, config):
    """
    Function to segment hydrocarbon rocks under UV luminescence.
    Uses Color Thresholding and Area Filtering according to the article:
    "Novel Lithology Identification Method for Drilling Cuttings Under PDC Bit".
    
    Args:
        image_rgb (np.ndarray): The input RGB image.
        config (dict): The color thresholds configuration dictionary.
        
    Returns:
        np.ndarray: The final binary mask combined for all detected oil classes.
        np.ndarray: The segmented RGB image.
    """
    # Step 1: Convert RGB to HSV
    # Disassociating Hue and Value strictly isolates the color from light intensity variations and tray reflections.
    hsv_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)

    # Initialize a globally grouped initial mask to handle multiple colors
    combined_initial_mask = np.zeros(hsv_image.shape[:2], dtype=np.uint8)

    # Step 2: Extract bounds and apply Color Thresholding dynamically for each class
    for oil_class, bounds in config.items():
        lower_bound = bounds["lower"]
        upper_bound = bounds["upper"]
        
        # Segment the specific color class
        class_mask = cv2.inRange(hsv_image, lower_bound, upper_bound)
        
        # Combine the masks
        combined_initial_mask = cv2.bitwise_or(combined_initial_mask, class_mask)

    # Step 3: Area Filtering
    # Find contours on the merged initial mask
    contours, _ = cv2.findContours(combined_initial_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Prepare an empty canvas for the final cleaned mask
    final_mask = np.zeros_like(combined_initial_mask)
    
    # Adhere strictly to the academic constraint: Area (P) >= 100 pixels.
    # Discard non-hydrocarbon microscopic artifacts.
    for contour in contours:
        area = cv2.contourArea(contour)
        if area >= 10:  
            # Retain robust hydrocarbon indications
            cv2.drawContours(final_mask, [contour], -1, 255, thickness=cv2.FILLED)

    # Apply the mask to the original image to visualize the isolated rocks
    segmented_rgb = cv2.bitwise_and(image_rgb, image_rgb, mask=final_mask)

    return final_mask, segmented_rgb

In [ ]:
def visualize_segmentation(image_rgb):
    final_mask, segmented_rock = segment_hydrocarbon_uv(image_rgb, COLOR_THRESHOLDS)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
    axes = axes.flatten() 

    axes[0].imshow(image_rgb)
    axes[0].set_title("Original UV Photo")
    axes[0].axis('off')

    axes[1].imshow(final_mask, cmap='gray')

    axes[1].set_title(f"Binary Mask (Area ≥ {pixel_threshold} px)")
    axes[1].axis('off')

    axes[2].imshow(segmented_rock)
    axes[2].set_title("Segmented Hydrocarbon Rocks")
    axes[2].axis('off')

    total_px = image_rgb.shape[0] * image_rgb.shape[1]
    detected_px = np.sum(final_mask > 0)
    ratio = detected_px / total_px * 100

    fig.suptitle(f"Oil-bearing area proportion: {ratio:.2f}%", fontsize=13, y=0.96)

    display(fig)
    plt.close(fig)  

In [ ]:
# -------------------------------------------------------------
# DYNAMIC IMAGE UPLOADER WIDGET
# -------------------------------------------------------------

# Widget
upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Pilih Gambar UV',
    button_style='info'
)

process_btn = widgets.Button(
    description='Segmentasi',
    button_style='success',
    icon='microscope'
)

output_canvas = widgets.Output()

def on_process_clicked(b):
    with output_canvas:
        clear_output(wait=True)
        if not upload_widget.value:
            print("⚠️  Upload gambar terlebih dahulu.")
            return
        try:
            uploaded_file = upload_widget.value[0]
            image_bytes = uploaded_file['content']
            image_pil = Image.open(io.BytesIO(image_bytes)).convert('RGB')
            image_rgb = np.array(image_pil)

            visualize_segmentation(image_rgb)

        except Exception as e:
            print(f"❌ Error: {e}")
            print("Pastikan file gambar RGB yang valid.")

process_btn.on_click(on_process_clicked)

display(widgets.VBox([
    widgets.HTML(
        "<h3>UV Fluorescence Segmentation</h3>"
        "<p>1. Select the image of the rock cut under UV light.<br>"
        "2. Click the <b>Segmentation</b> button to view the results.</p>"
    ),
    upload_widget,
    process_btn,
    output_canvas
]))